# Benchmark Answer Generator Comparison

This notebook demonstrates how to use the `run_benchmarks` API to evaluate and compare different `AnswerGenerator` implementations. It runs both the `fix_error` and `api_understanding` benchmark suites and visualizes the pass rate of each generator.

In [ ]:
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import the necessary components from the benchmark framework
from benchmarks import test_rig
from benchmarks.answer_generators import (
    GroundTruthAnswerGenerator,
    TrivialAnswerGenerator,
)

# Set a nice style for the plots
sns.set_theme(style="whitegrid")

## 1. Configure and Run the Benchmarks

First, we define which benchmark suites and which answer generators we want to run. For this comparison, we'll run all available suites and compare the `GroundTruthAnswerGenerator` (which should always be perfect) against the `TrivialAnswerGenerator` (which provides a simple, often incorrect, answer).

In [ ]:
async def run_comparison():
    """Sets up and runs the benchmark comparison."""
    print("Configuring benchmark run...")
    
    benchmark_suites = [
        "benchmarks/benchmark_definitions/fix_error_benchmarks.yaml",
        "benchmarks/benchmark_definitions/api_understanding_benchmarks.yaml",
    ]
    
    answer_generators = [
        GroundTruthAnswerGenerator(), 
        TrivialAnswerGenerator()
    ]
    
    print("Executing benchmarks...")
    summary_df = await test_rig.run_benchmarks(benchmark_suites, answer_generators)
    
    return summary_df

### Executing the Asynchronous Function

Because Jupyter Notebooks run on an active `asyncio` event loop, we cannot use `asyncio.run()`. Instead, we call our async function directly with `await`.

In [ ]:
# Run the asynchronous benchmark function using top-level await
summary_results = await run_comparison()

## 2. Display and Visualize the Results

The `run_benchmarks` function returns a clean pandas DataFrame with the pass/total statistics for each generator. We can print this DataFrame and then use it to create a bar chart for easy comparison.

In [ ]:
print("--- Benchmark Summary ---")
print(summary_results)

# Create the visualization
plt.figure(figsize=(10, 6))
sns.barplot(x=summary_results.index, y=summary_results['pass_rate'])

plt.title('Answer Generator Performance', fontsize=16)
plt.xlabel('Answer Generator', fontsize=12)
plt.ylabel('Pass Rate', fontsize=12)
plt.ylim(0, 1.1) # Set y-axis limit to be just above 1.0 for clarity
plt.xticks(rotation=15) # Rotate labels slightly for better readability

# Add the pass rate value on top of each bar
for index, row in summary_results.iterrows():
    plt.text(row.name, row.pass_rate + 0.02, f'{row.pass_rate:.2%}', color='black', ha="center")

plt.show()